# Example 4.4 / 5: Time Periods: Inventory Balance

Companion notebook for **4.4 Ordered Sets and Time Periods** in the AMPL lab Documentation, section "Time Periods: Inventory Balance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Time Periods: Inventory Balance

Coils sell for more in the second week (35 against 30), but that week's market takes only 2500
tons, which is more than one week's hours can make. So 1100 tons of coils made in week `27sep`
go into inventory and are sold in week `04oct`: `Inv['coils','27sep']` appears in
`Balance['coils','04oct']` as `Inv[p,prev(t)]`.

> **Pitfall: `prev` of the first period**
> Indexing `Balance` over every week makes AMPL compute `prev` of the first week, which
> doesn't exist. The model reads without complaint, and the error comes when the constraint is
> generated, at the solve. From Python, `ampl.solve()` logs the error, and
> `ampl.solve_result` is `?`: nothing was solved.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set WEEKS ordered;
param inv0 {PROD} >= 0;
var Make {PROD, WEEKS} >= 0;
var Inv {PROD, WEEKS} >= 0;
var Sell {PROD, WEEKS} >= 0;

subject to Balance {p in PROD, t in WEEKS}:
    Make[p,t] + Inv[p,prev(t)] = Sell[p,t] + Inv[p,t];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands"]
ampl.set["WEEKS"] = ["27sep", "04oct"]
ampl.param["inv0"] = {"bands": 10}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)

[ERROR] 
	Error executing "solve" command:
	error processing constraint Balance['bands','27sep']:
		can't compute prev('27sep', WEEKS) -- '27sep' is the first member


?


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`, where `solve` prints the message:

In [6]:
%%writefile data.dat
set PROD := bands;
set WEEKS := 27sep 04oct;
param inv0 := bands 10;

Writing data.dat


In [7]:
# A fresh session for the pure-AMPL run. amplpy reports the failed solve of run.py again
# at the next call, so that first call is allowed to fail.
try:
    ampl.eval("reset;")
except Exception:
    pass
ampl.eval(r"""
reset;
model model.mod;
data data.dat;
""")

In [8]:
# model.run: this step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
option solver highs;
solve;
""")
except Exception as error:
    print(error)

[ERROR] 
	Error executing "solve" command:
	error processing constraint Balance['bands','27sep']:
		can't compute prev('27sep', WEEKS) -- '27sep' is the first member


Error executing "solve" command:
error processing constraint Balance['bands','27sep']:
	can't compute prev('27sep', WEEKS) -- '27sep' is the first member

For support/feedback go to https://discuss.ampl.com or e-mail <support@ampl.com>

